# Counted: every headline number, recomputed offline

No AWS credentials, no model calls, no cost. This notebook reads the saved per-case predictions in `evals/results/` (413 benchmark cases from the final run, plus 10 near misses) and recomputes the accuracy tables, 95% Wilson intervals, exact McNemar tests against the baselines, each seat's accuracy alone, Claude–Nova agreement, and false links on the no-link deaths.

Run it with `uv run --with jupyter jupyter notebook notebooks/reproduce.ipynb`, or `uv run python -m evals.stats` for the same numbers in a terminal.

In [1]:
import os, sys
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, '.')
from evals import stats
stats.main()


Hurricane Irma (CDC labels) (n=129)
  direct_only   11/129    8.5%  95% CI 4.8% to 14.6%
  single_llm    68/129   52.7%  95% CI 44.1% to 61.1%
  rules        121/129   93.8%  95% CI 88.2% to 96.8%
  claude       129/129  100.0%  95% CI 97.1% to 100.0%
  nova         127/129   98.5%  95% CI 94.5% to 99.6%
  panel        129/129  100.0%  95% CI 97.1% to 100.0%
  panel vs direct_only: 118 vs 0 discordant, exact p = 6e-36
  panel vs single_llm: 61 vs 0 discordant, exact p = 8.7e-19

Winter Storm Uri (Texas DSHS labels) (n=246)
  direct_only  148/246   60.2%  95% CI 53.9% to 66.1%
  single_llm   173/246   70.3%  95% CI 64.3% to 75.7%
  rules        214/246   87.0%  95% CI 82.2% to 90.6%
  claude       229/246   93.1%  95% CI 89.2% to 95.6%
  nova         222/246   90.2%  95% CI 85.9% to 93.4%
  panel        229/246   93.1%  95% CI 89.2% to 95.6%
  panel vs direct_only: 81 vs 0 discordant, exact p = 8.3e-25
  panel vs single_llm: 56 vs 0 discordant, exact p = 2.8e-17

CDC worked examples (n

## The labels behind these numbers

Irma and Uri labels are the government's own (CDC MMWR 2018; Texas DSHS 2021), one per official death; `data/check_tables.py` re-adds every transcribed table against its source. 19 of the 22 CDC worked-example labels are implied by the guide rather than stated, and the 26 no-link deaths were written by the author; `data/SOURCES.md` says which is which.

In [2]:
import json, collections
rows = [json.loads(l) for l in open('evals/results/full-v3/predictions.jsonl', encoding='utf-8')]
print(collections.Counter(r['set'] for r in rows))
[r for r in rows if r['set'] == 'irma_replay'][0]['counted']['causal_chain']

Counter({'uri_replay': 246, 'irma_replay': 129, 'cdc_gold': 22, 'negatives': 8, 'pressure': 8})


"Hurricane Irma produced storm surge that inundated the decedent's residence in Hillsborough County, FL; unable to escape the rising water, the decedent drowned."